# Wanderings through duckdb and polars

Findings: 
1. gosh this duckdb and polars stuff is fast.
2. now we get to figure out the useful views we need and we can work on the fast because parquet has modern bindings in so many places, from R to python.

In [1]:
import duckdb 
import polars as pl
from pathlib import Path



In [2]:
print("duckDB Version =", duckdb.__version__)
print("Polars Build Info =", pl.build_info())
print("Polars Thread Pool Size =", pl.thread_pool_size())


duckDB Version = 1.5.6
Polars Build Info = {'version': '1.44.2'}
Polars Thread Pool Size = 8


In [3]:
# Various paths to data sets in the repo

NATIONAL_CSB_PARQUET_PATH = Path("../out/national1825.parquet")
CSB_LAND_COVER_CODES_PARQUET_PATH = Path("../useful-views/land_cover_codes.parquet")

In [4]:
national = None
national = duckdb.connect()
national.execute("LOAD spatial")

In [5]:
# Excluding `Shape` as that is a multipolygon column.
# It is not easily rendered and not needed for this example.

"""
Also note, displaying this on your local (VS Code) should be pretty easy, but displaying on github through the  
html render of the notebook may require that your browser is gigantically wide, or text super small. 
On my 27 inch display its readable, but I for sure have my glasses on :) 

"""

national.sql(
    """
    SELECT * EXCLUDE Shape
    FROM read_parquet($path)
    LIMIT 20
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).show()



┌──────────┬─────────────────┬──────────┬────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬──────────┬─────────┬─────────┬──────────┬───────────────────┬──────────────┬──────────┬────────────────────┬────────────────────┐
│ OBJECTID │      CSBID      │ CSBYEARS │      CSBACRES      │ CDL2018 │ CDL2019 │ CDL2020 │ CDL2021 │ CDL2022 │ CDL2023 │ CDL2024 │ CDL2025 │ STATEFIPS │ STATEASD │   ASD   │  CNTY   │ CNTYFIPS │     INSIDE_X      │   INSIDE_Y   │ ORIG_FID │    Shape_Length    │     Shape_Area     │
│  int64   │     varchar     │ varchar  │       double       │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  varchar  │ varchar  │ varchar │ varchar │ varchar  │      double       │    double    │  int32   │       double       │       double       │
├──────────┼─────────────────┼──────────┼────────────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼───

In [6]:

# if controlled out of the notebook flow - uncomment to see the nature of the multipolygon.

if 0:
    national.sql(
        """
        SELECT Shape
        FROM read_parquet($path)
        LIMIT 20
        """,
        params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
    ).show()

In [7]:
shape_sample = None
shape_sample = national.sql(
    """
    SELECT ST_AsText(Shape) AS Shape
    FROM read_parquet($path)
    LIMIT 20
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).pl().slice(17, 1)

#just to give the reader a sense of the specific shape at this position...



In [8]:
print(shape_sample)

shape: (1, 1)
┌─────────────────────────────────┐
│ Shape                           │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ MULTIPOLYGON (((-663619.003499… │
└─────────────────────────────────┘


In [9]:
csb_codes = None
csb_codes = duckdb.connect()
csb_codes.execute("LOAD spatial")

In [10]:
csb_codes.sql(
    """
    SELECT *
    FROM read_parquet($path)
    LIMIT 20
    """,
    params={"path": str(CSB_LAND_COVER_CODES_PARQUET_PATH)},
)

┌─────────────────┬──────────────────────────┐
│ land_cover_code │     land_cover_name      │
│      int64      │         varchar          │
├─────────────────┼──────────────────────────┤
│               0 │ Background               │
│               1 │ Corn                     │
│               2 │ Cotton                   │
│               3 │ Rice                     │
│               4 │ Sorghum                  │
│               5 │ Soybeans                 │
│               6 │ Sunflower                │
│              10 │ Peanuts                  │
│              11 │ Tobacco                  │
│              12 │ Sweet Corn               │
│              13 │ Pop or Orn Corn          │
│              14 │ Mint                     │
│              21 │ Barley                   │
│              22 │ Durum Wheat              │
│              23 │ Spring Wheat             │
│              24 │ Winter Wheat             │
│              25 │ Other Small Grains       │
│            

In [12]:
csb_field_year = None
csb_field_year = national.sql(
    """
    SELECT * EXCLUDE Shape
    FROM read_parquet($path)
    LIMIT 10
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).show()

┌──────────┬─────────────────┬──────────┬────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬──────────┬─────────┬─────────┬──────────┬───────────────────┬──────────────┬──────────┬────────────────────┬────────────────────┐
│ OBJECTID │      CSBID      │ CSBYEARS │      CSBACRES      │ CDL2018 │ CDL2019 │ CDL2020 │ CDL2021 │ CDL2022 │ CDL2023 │ CDL2024 │ CDL2025 │ STATEFIPS │ STATEASD │   ASD   │  CNTY   │ CNTYFIPS │     INSIDE_X      │   INSIDE_Y   │ ORIG_FID │    Shape_Length    │     Shape_Area     │
│  int64   │     varchar     │ varchar  │       double       │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  varchar  │ varchar  │ varchar │ varchar │ varchar  │      double       │    double    │  int32   │       double       │       double       │
├──────────┼─────────────────┼──────────┼────────────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼───

In [13]:
national.sql(
    """
    SELECT * EXCLUDE Shape
    FROM read_parquet($path)
    LIMIT 200
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).show()

┌──────────┬─────────────────┬──────────┬────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬──────────┬─────────┬─────────┬──────────┬───────────────────┬──────────────┬──────────┬────────────────────┬────────────────────┐
│ OBJECTID │      CSBID      │ CSBYEARS │      CSBACRES      │ CDL2018 │ CDL2019 │ CDL2020 │ CDL2021 │ CDL2022 │ CDL2023 │ CDL2024 │ CDL2025 │ STATEFIPS │ STATEASD │   ASD   │  CNTY   │ CNTYFIPS │     INSIDE_X      │   INSIDE_Y   │ ORIG_FID │    Shape_Length    │     Shape_Area     │
│  int64   │     varchar     │ varchar  │       double       │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  varchar  │ varchar  │ varchar │ varchar │ varchar  │      double       │    double    │  int32   │       double       │       double       │
├──────────┼─────────────────┼──────────┼────────────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼───

In [14]:
csb_field = None
csb_field = national.sql(
    """
    SELECT CSBID, CSBACRES, INSIDE_X, INSIDE_Y, Shape
    FROM read_parquet($path)
    LIMIT 10
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).show()





┌─────────────────┬────────────────────┬───────────────────┬──────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [17]:
national.sql(
    """
 SELECT
    CSBID AS csb_id,
    STATEFIPS || CNTYFIPS AS county_geoid,
    CSBACRES AS csb_acres,
    INSIDE_X AS inside_x,
    INSIDE_Y AS inside_y,
    Shape AS geometry
    FROM read_parquet($path)
    ORDER BY csb_id
    LIMIT 10
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).show()





┌─────────────────┬──────────────┬────────────────────┬──────────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [22]:
tmp_fy = None

tmp_fy = national.sql(
    """
    SELECT
      CSBID AS csb_id,
      CAST(substr(cdl_column, 4) AS SMALLINT) AS year,
    CAST(land_cover_code AS SMALLINT) AS land_cover_code
FROM (
    UNPIVOT (SELECT CSBID, COLUMNS('^CDL\\d{4}$') FROM read_parquet($path)
)
    ON COLUMNS('^CDL\\d{4}$')
    INTO NAME cdl_column VALUE land_cover_code
)
ORDER BY csb_id, year
    LIMIT 1000
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
)

In [25]:
tmp_fy

┌─────────────────┬───────┬─────────────────┐
│     csb_id      │ year  │ land_cover_code │
│     varchar     │ int16 │      int16      │
├─────────────────┼───────┼─────────────────┤
│ 011825000016732 │  2018 │               2 │
│ 011825000016732 │  2019 │               2 │
│ 011825000016732 │  2020 │               2 │
│ 011825000016732 │  2021 │               1 │
│ 011825000016732 │  2022 │               5 │
│ 011825000016732 │  2023 │             176 │
│ 011825000016732 │  2024 │             176 │
│ 011825000016732 │  2025 │             176 │
│ 011825000016733 │  2018 │               5 │
│ 011825000016733 │  2019 │               5 │
│        ·        │    ·  │               · │
│        ·        │    ·  │               · │
│        ·        │    ·  │               · │
│ 011825000016855 │  2024 │             176 │
│ 011825000016855 │  2025 │             176 │
│ 011825000016856 │  2018 │              37 │
│ 011825000016856 │  2019 │              37 │
│ 011825000016856 │  2020 │       

In [ ]:
csb_field.limit(10).show()

In [15]:
national.sql(
    """
    SELECT * EXCLUDE Shape
    FROM read_parquet($path)
    WHERE CSBID = 351825000000001
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).show()

┌──────────┬─────────────────┬──────────┬───────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬──────────┬─────────┬─────────┬──────────┬──────────────┬──────────────┬──────────┬────────────────────┬───────────────────┐
│ OBJECTID │      CSBID      │ CSBYEARS │     CSBACRES      │ CDL2018 │ CDL2019 │ CDL2020 │ CDL2021 │ CDL2022 │ CDL2023 │ CDL2024 │ CDL2025 │ STATEFIPS │ STATEASD │   ASD   │  CNTY   │ CNTYFIPS │   INSIDE_X   │   INSIDE_Y   │ ORIG_FID │    Shape_Length    │    Shape_Area     │
│  int64   │     varchar     │ varchar  │      double       │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  varchar  │ varchar  │ varchar │ varchar │ varchar  │    double    │    double    │  int32   │       double       │      double       │
├──────────┼─────────────────┼──────────┼───────────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼───────────┼──────────┼──

In [16]:
national.sql(
    """
    SELECT * EXCLUDE Shape
    FROM read_parquet($path)
    WHERE CSBID = 351825000000008
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).show()

┌──────────┬─────────────────┬──────────┬────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬──────────┬─────────┬─────────┬──────────┬──────────────┬──────────────┬──────────┬────────────────────┬────────────────────┐
│ OBJECTID │      CSBID      │ CSBYEARS │      CSBACRES      │ CDL2018 │ CDL2019 │ CDL2020 │ CDL2021 │ CDL2022 │ CDL2023 │ CDL2024 │ CDL2025 │ STATEFIPS │ STATEASD │   ASD   │  CNTY   │ CNTYFIPS │   INSIDE_X   │   INSIDE_Y   │ ORIG_FID │    Shape_Length    │     Shape_Area     │
│  int64   │     varchar     │ varchar  │       double       │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  varchar  │ varchar  │ varchar │ varchar │ varchar  │    double    │    double    │  int32   │       double       │       double       │
├──────────┼─────────────────┼──────────┼────────────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼───────────┼──────

In [26]:
!pwd

/Users/jameshorine/repos/useful-views/crop-sequence-boundaries/sandbox


In [27]:
pl.read_parquet("../useful-views/csb_2025/county.parquet")

county_geoid,state_fips,county_fips,county_name,asd_code
str,str,str,str,str
"""01001""","""01""","""001""","""Autauga""","""0140"""
"""01003""","""01""","""003""","""Baldwin""","""0150"""
"""01005""","""01""","""005""","""Barbour""","""0160"""
"""01007""","""01""","""007""","""Bibb""","""0130"""
"""01009""","""01""","""009""","""Blount""","""0120"""
…,…,…,…,…
"""56037""","""56""","""037""","""Sweetwater""","""5640"""
"""56039""","""56""","""039""","""Teton""","""5630"""
"""56041""","""56""","""041""","""Uinta""","""5630"""


In [28]:
pl.read_parquet("../useful-views/csb_2025/csb_field_year.parquet")

csb_id,year,land_cover_code
str,i16,i16
"""011825000016732""",2018,2
"""011825000016732""",2019,2
"""011825000016732""",2020,2
"""011825000016732""",2021,1
"""011825000016732""",2022,5
…,…,…
"""561825015410849""",2021,36
"""561825015410849""",2022,36
"""561825015410849""",2023,36


In [ ]:
fy_p = None
fy_p = pl.read_parquet("../useful-views/csb_2025/c")